# 03: Smoke Test + Baselines + ORA

Workhorse notebook. Runs `grounded-vla smoke`, then evaluates the three agents
(ReAct + Mistral, single-shot LLaVA, ORA + LLaVA) across ScienceQA, the
synthetic sample, and Mind2Web. All artifacts go to
`/content/drive/MyDrive/grounded_vla/runs/<agent>__<dataset>/`.

**Runtime:** A100 GPU **strongly recommended** (Pro+ default).

**Estimated wallclock:** ~60-90 min on A100 for the full sweep.
**Estimated compute units:** ~15-20 (well within Pro+'s 500/month).

## Pro+ tip: background execution

Tools → Settings → check **'Run after disconnecting'**. With this on, you can
close the tab; Colab keeps the runtime alive for up to 24 hours and the
checkpointed `EvalRunner` will keep writing per-task results to Drive. Re-open
the notebook later and the results will be waiting in Drive.

If a session does die mid-sweep, just re-run — `resume=True` skips already-
completed tasks via the trajectory JSONs already on Drive.

In [1]:
# Mount Google Drive. The first run prompts for OAuth; subsequent runs
# in the same session reuse the token automatically.
from google.colab import drive
drive.mount('/content/drive')

import os, pathlib
ROOT = pathlib.Path('/content/drive/MyDrive/grounded_vla')
ROOT.mkdir(parents=True, exist_ok=True)
print('drive root:', ROOT)

Mounted at /content/drive
drive root: /content/drive/MyDrive/grounded_vla


In [2]:
# Verify accelerator. Pro+ should give you A100 most of the time; if you
# see T4, change Runtime -> Change runtime type -> A100 GPU and re-run.
import subprocess
subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total,memory.free',
                '--format=csv'])

CompletedProcess(args=['nvidia-smi', '--query-gpu=name,memory.total,memory.free', '--format=csv'], returncode=0)

In [3]:
# Clone repo if absent, then always pull to get the latest fixes.
# With an editable install (-e) a pull is all that's needed — no reinstall.
REPO_URL = 'https://github.com/KarthikRed2000/grounded_vla.git'
import os, subprocess
if not os.path.exists('/content/grounded_vla'):
    subprocess.run(['git', 'clone', REPO_URL, '/content/grounded_vla'], check=True)
subprocess.run(['git', '-C', '/content/grounded_vla', 'pull'], check=True)
%cd /content/grounded_vla

/content/grounded_vla


In [4]:
# Install repo + GPU stack. Quiet to keep the cell output sane.
!pip install -q -e .[gpu,data]

  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 43.8 MB/s eta 0:00:00
  Building editable for grounded_vla (pyproject.toml) ... done


In [5]:
# Point HF cache at the Drive-mounted weights so transformers loads from there.
import os
WEIGHTS = '/content/drive/MyDrive/grounded_vla/hf_cache'
DATA    = '/content/drive/MyDrive/grounded_vla/data'
os.environ['HF_HOME'] = WEIGHTS
os.environ['TRANSFORMERS_CACHE'] = WEIGHTS
os.environ['TRANSFORMERS_OFFLINE'] = '1'
os.environ['HF_DATASETS_OFFLINE'] = '1'
assert os.path.isdir(WEIGHTS), 'run notebook 01 first'
assert os.path.isdir(DATA),    'run notebook 02 first'

In [6]:
# Smoke test: pure-mock pipeline. Should print 'smoke ok' in <5 seconds.
!grounded-vla smoke

smoke ok: agent=ora-vlm, n=200, success=0.01, mean_steps=1.45


In [7]:
# Build the three agents. On A100 we can use the default GenerationConfig
# without any concessions to memory.
from grounded_vla.agents import ORAAgent, ReActAgent, SingleShotVLMAgent
from grounded_vla.backends import make_backend
from grounded_vla.backends.base import GenerationConfig

WEIGHTS = '/content/drive/MyDrive/grounded_vla/hf_cache'

def build_react():
    backend = make_backend({
        'kind': 'mistral',
        'model_id': f'{WEIGHTS}/Mistral-7B-Instruct-v0.2',
        'device': 'cuda',
        'quantize': '4bit',
    })
    return ReActAgent(backend, GenerationConfig(max_new_tokens=384, temperature=0.1))

def build_single_shot():
    backend = make_backend({
        'kind': 'llava',
        'model_id': f'{WEIGHTS}/llava-v1.6-mistral-7b-hf',
        'device': 'cuda',
        'quantize': '4bit',
    })
    return SingleShotVLMAgent(backend, GenerationConfig(max_new_tokens=256, temperature=0.0))

def build_ora():
    backend = make_backend({
        'kind': 'llava',
        'model_id': f'{WEIGHTS}/llava-v1.6-mistral-7b-hf',
        'device': 'cuda',
        'quantize': '4bit',
    })
    return ORAAgent(backend, GenerationConfig(max_new_tokens=384, temperature=0.1))

In [8]:
# Build the three datasets pointing at the Drive-mounted data.
from grounded_vla.data import make_dataset
DATA = '/content/drive/MyDrive/grounded_vla/data'

def scienceqa(limit=200):
    return make_dataset({
        'kind': 'scienceqa',
        'jsonl_path': f'{DATA}/scienceqa/test.jsonl',
        'images_dir': f'{DATA}/scienceqa',
        'limit': limit,
    })

def synthetic():
    return make_dataset({
        'kind': 'jsonl',
        'path': f'{DATA}/synthetic_sample/synthetic_sample.jsonl',
        'source': 'synthetic',
    })

In [9]:
# Generic eval runner with checkpointing -> Drive. Resumable across sessions.
from grounded_vla.eval import EvalRunner
from pathlib import Path

RUNS = Path('/content/drive/MyDrive/grounded_vla/runs')
RUNS.mkdir(parents=True, exist_ok=True)

def run(agent_name, agent, ds_name, dataset, **kw):
    out = RUNS / f'{agent_name}__{ds_name}'
    runner = EvalRunner(agent)
    res = runner.evaluate(
        dataset, save_dir=out, checkpoint_every=5, resume=True, **kw
    )
    print(f'{agent_name} on {ds_name}: '
          f'success={res.task_completion_rate:.3f} '
          f'mean_steps={res.mean_steps:.2f} '
          f'errors={res.error_breakdown}')
    return res

## ReAct + Mistral (Baseline 1)

Text-only. ~1-2 sec/task on A100 with 4-bit Mistral. ~3 minutes total.

In [10]:
react = build_react()
_ = run('react_mistral', react, 'scienceqa',  scienceqa(limit=200))
_ = run('react_mistral', react, 'synthetic',  synthetic())
react.backend.close()

[05/05/26 18:28:10] INFO     INFO:grounded_vla.eval.runner:resuming: 200 tasks already on disk

react_mistral on scienceqa: success=0.235 mean_steps=1.16 errors={'visual_misgrounding': 105, 'action_parsing_failure': 25, 'none': 47, 'reasoning_error': 22, 'truncated': 1}


[05/05/26 18:28:16] INFO     INFO:grounded_vla.eval.runner:resuming: 46 tasks already on disk

[05/05/26 18:28:43] INFO     INFO:numexpr.utils:NumExpr defaulting to 12 threads.

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

...

Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.


[05/05/26 18:59:05] INFO     INFO:grounded_vla.eval.runner:checkpoint @ task 195 ->                                
                             /content/drive/MyDrive/grounded_vla/runs/react_mistral__synthetic

Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.


[05/05/26 18:59:53] INFO     INFO:grounded_vla.eval.runner:checkpoint @ task 200 ->                                
                             /content/drive/MyDrive/grounded_vla/runs/react_mistral__synthetic

react_mistral on synthetic: success=0.080 mean_steps=1.19 errors={'none': 16, 'reasoning_error': 66, 'action_parsing_failure': 45, 'truncated': 35, 'visual_misgrounding': 38}


## Single-shot LLaVA (Baseline 2)

VLM, one call per task. Tests H1 (visual grounding helps).

In [11]:
ss = build_single_shot()
_ = run('single_shot_llava', ss, 'scienceqa',  scienceqa(limit=200))
_ = run('single_shot_llava', ss, 'synthetic',  synthetic())
ss.backend.close()

[05/05/26 18:59:55] INFO     INFO:grounded_vla.eval.runner:resuming: 200 tasks already on disk

single_shot_llava on scienceqa: success=0.465 mean_steps=1.00 errors={'none': 93, 'visual_misgrounding': 53, 'reasoning_error': 28, 'action_parsing_failure': 26}


[05/05/26 18:59:58] INFO     INFO:grounded_vla.eval.runner:resuming: 46 tasks already on disk

The image processor of type `LlavaNextImageProcessor` is now loaded as a fast processor by default, even if the model checkpoint was saved with a slow processor. This is a breaking change and may produce slightly different outputs. To continue using the slow processor, instantiate this class with `use_fast=False`. 


Loading weights:   0%|          | 0/687 [00:00<?, ?it/s]

...

Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.


[05/05/26 19:15:43] INFO     INFO:grounded_vla.eval.runner:checkpoint @ task 195 ->                                
                             /content/drive/MyDrive/grounded_vla/runs/single_shot_llava__synthetic

Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.


[05/05/26 19:16:10] INFO     INFO:grounded_vla.eval.runner:checkpoint @ task 200 ->                                
                             /content/drive/MyDrive/grounded_vla/runs/single_shot_llava__synthetic

single_shot_llava on synthetic: success=0.315 mean_steps=1.00 errors={'none': 63, 'visual_misgrounding': 71, 'action_parsing_failure': 40, 'reasoning_error': 26}


## ORA + LLaVA (Our Method)

VLM with the Observe-Reason-Act loop and per-step visual re-encoding. Tests H2.
This is the single longest cell in the project (~30-50 min on A100).

In [12]:
ora = build_ora()
_ = run('ora_llava', ora, 'scienceqa',  scienceqa(limit=200))
_ = run('ora_llava', ora, 'synthetic',  synthetic())
ora.backend.close()

[05/05/26 19:16:12] INFO     INFO:grounded_vla.eval.runner:resuming: 200 tasks already on disk

ora_llava on scienceqa: success=0.565 mean_steps=1.23 errors={'none': 113, 'visual_misgrounding': 66, 'reasoning_error': 14, 'truncated': 7}


[05/05/26 19:16:15] INFO     INFO:grounded_vla.eval.runner:resuming: 46 tasks already on disk

Loading weights:   0%|          | 0/687 [00:00<?, ?it/s]

Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.


...

Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.


[05/05/26 19:46:33] INFO     INFO:grounded_vla.eval.runner:checkpoint @ task 195 ->                                
                             /content/drive/MyDrive/grounded_vla/runs/ora_llava__synthetic

Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.


[05/05/26 19:47:06] INFO     INFO:grounded_vla.eval.runner:checkpoint @ task 200 ->                                
                             /content/drive/MyDrive/grounded_vla/runs/ora_llava__synthetic

ora_llava on synthetic: success=0.290 mean_steps=1.71 errors={'none': 58, 'visual_misgrounding': 35, 'truncated': 94, 'action_parsing_failure': 10, 'reasoning_error': 3}


In [13]:
# Aggregate results table for the report.
import json
from pathlib import Path
rows = []
for d in sorted(Path('/content/drive/MyDrive/grounded_vla/runs').iterdir()):
    s = json.loads((d / 'summary.json').read_text())
    rows.append((d.name, s['n_tasks'], s['task_completion_rate'], s['mean_steps'], s['error_breakdown']))
for r in rows:
    print(f'{r[0]:30s}  n={r[1]:3d}  success={r[2]:.3f}  steps={r[3]:.2f}  errors={r[4]}')

ora_llava__scienceqa            n=200  success=0.565  steps=1.23  errors={'none': 113, 'visual_misgrounding': 66, 'reasoning_error': 14, 'truncated': 7}
ora_llava__synthetic            n=200  success=0.290  steps=1.71  errors={'none': 58, 'visual_misgrounding': 35, 'truncated': 94, 'action_parsing_failure': 10, 'reasoning_error': 3}
react_mistral__scienceqa        n=200  success=0.235  steps=1.16  errors={'visual_misgrounding': 105, 'action_parsing_failure': 25, 'none': 47, 'reasoning_error': 22, 'truncated': 1}
react_mistral__synthetic        n=200  success=0.080  steps=1.19  errors={'none': 16, 'reasoning_error': 66, 'action_parsing_failure': 45, 'truncated': 35, 'visual_misgrounding': 38}
single_shot_llava__scienceqa    n=200  success=0.465  steps=1.00  errors={'none': 93, 'visual_misgrounding': 53, 'reasoning_error': 28, 'action_parsing_failure': 26}
single_shot_llava__synthetic    n=200  success=0.315  steps=1.00  errors={'none': 63, 'visual_misgrounding': 71, 'action_parsing_fail

## Done

Results live at `/content/drive/MyDrive/grounded_vla/runs/`. The H1 and H2
comparisons drop straight out of the table above:

- **H1 (vision helps):** compare `react_mistral` vs `single_shot_llava` per dataset.
- **H2 (loop helps):** compare `single_shot_llava` vs `ora_llava` per dataset.

Move on to `04_lora_finetune.ipynb` for the H3 stretch goal.